# Лабораторна робота №4. AI-підкріплена дата аналітика

Виконала: Ткаченко Ганна Олексіївна, група ІК-32

У цій роботі я зробила асистента для аналітика у вигляді набору функцій у ноутбуці. Усі числа рахує Pandas, а мовна модель Gemini отримує вже пораховані дані і пише висновки, гіпотези або код. Після кожної відповіді асистент перевіряє, чи є числа з тексту моделі серед порахованих, тому вигадане число одразу видно.

Функції асистента:
* auto_eda() – автоматичний EDA таблиці і висновки про якість даних;
* suggest_hypotheses() – гіпотези, які спираються на цифри з даних;
* ask() – питання звичайною мовою, код Pandas і результат;
* explain_anomaly() – пояснення, які сегменти найбільше вплинули на зміну метрики;
* executive_summary() – короткий текст для продакта;
* check_numbers() – моя власна функція для перевірки чисел у відповідях моделі.

Для тесту я взяла дані з лабораторної №2: 4030 користувачів, 42 420 подій і 323 підписки.

## 1. Налаштування

Я використовую безкоштовний API Gemini. Ключ зберігаю в Colab Secrets під назвою GEMINI_API_KEY.

In [76]:
!pip install -q google-genai

In [77]:
import os
import re
import json
import time
import zipfile
import builtins
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

MODELS = ['gemini-3.8-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest']


def get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('GEMINI_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        raise RuntimeError('Не знайдено GEMINI_API_KEY. Додайте ключ у Colab Secrets або в змінну середовища.')
    return key

Функція llm_json надсилає запит до моделі і повертає відповідь у форматі JSON. Я вмикаю JSON mode, бо з JSON легко дістати потрібні поля і перевірити числа. Безкоштовні моделі Gemini іноді перевантажені або мають ліміт запитів. Тому функція пробує кілька моделей по черзі, а якщо всі зайняті, то вона чекає і пробує ще раз, до трьох кругів. Щоб запит не завис, я ставлю ліміт 2 хвилини на відповідь. Також я вимикаю режим "роздумів" моделі, бо для цих завдань він не потрібен, а якщо модель цього не підтримує, то запит іде без цього налаштування.

In [78]:
client = None
active_model = None


def parse_json(text):
    text = (text or '').strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.S)
        if match:
            return json.loads(match.group(0))
        raise ValueError('Відповідь моделі не є JSON')


def llm_json(system_prompt, user_prompt, temperature=0.2):
    global client, active_model
    from google import genai
    from google.genai import types
    if client is None:
        client = genai.Client(api_key=get_api_key(), http_options=types.HttpOptions(timeout=120000))
    models = [active_model] + [m for m in MODELS if m != active_model] if active_model else MODELS
    last_error = None
    for round_number in range(3):
        for model in models:
            for thinking_off in [True, False]:
                config = types.GenerateContentConfig(
                    system_instruction=system_prompt, temperature=temperature, response_mime_type='application/json',
                    thinking_config=types.ThinkingConfig(thinking_budget=0) if thinking_off else None)
                try:
                    response = client.models.generate_content(model=model, contents=user_prompt, config=config)
                    active_model = model
                    return parse_json(response.text)
                except Exception as error:
                    last_error = error
                    if '400' in str(error) and thinking_off:
                        continue
                    print(f'{model}: {str(error)[:120]}')
                    break
        print(f'Моделі зараз перевантажені, чекаю {20 * (round_number + 1)} секунд і пробую ще раз...')
        time.sleep(20 * (round_number + 1))
    raise RuntimeError(f'Модель не відповіла: {last_error}')

Перевіряю, що ключ працює і модель відповідає

In [79]:
print(llm_json('Відповідай лише JSON.', 'Поверни {"ok": true}'))
print('Модель:', active_model)

gemini-3.8-flash: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
gemini-flash-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
{'ok': True}
Модель: gemini-flash-lite-latest


## 2. Перевірка чисел

Це функція, де модель може впевнено написати число, якого немає в даних. Тому я дістаю всі числа з тексту моделі і шукаю кожне серед чисел, які порахував Pandas. Невеликі розбіжності через округлення допускаються, а дати, роки, номери фактів і цілі числа до 10 я не перевіряю.

In [80]:
NUMBER_RE = re.compile(r'(?<![\w.])-?\d{1,3}(?:[  ]\d{3})+(?:[.,]\d+)?|(?<![\w.])-?\d+(?:[.,]\d+)?')
DATE_RE = re.compile(r'\b\d{4}-\d{2}-\d{2}\b|\b\d{1,2}\.\d{2}(?:\.\d{4})?\b|\b(?:19|20)\d{2}\b|\bF\d+\b')


def extract_numbers(text):
    numbers = []
    for raw in NUMBER_RE.findall(DATE_RE.sub(' ', str(text))):
        try:
            numbers.append(float(raw.replace(' ', '').replace(' ', '').replace(',', '.')))
        except ValueError:
            pass
    return numbers


def collect_numbers(obj):
    if isinstance(obj, dict):
        return [n for k, v in obj.items() for n in collect_numbers(k) + collect_numbers(v)]
    if isinstance(obj, (list, tuple)):
        return [n for v in obj for n in collect_numbers(v)]
    if isinstance(obj, (int, float, np.integer, np.floating)) and not isinstance(obj, bool):
        return [float(obj)]
    if isinstance(obj, str):
        return extract_numbers(obj)
    return []


def check_numbers(text, source, extra=()):
    allowed = collect_numbers(source) + list(extra)
    not_found = []
    for number in extract_numbers(text):
        if abs(number) <= 10 and float(number).is_integer():
            continue
        if not any(abs(number - a) <= (0.55 if abs(a) < 100 else 0.01 * abs(a)) for a in allowed):
            not_found.append(number)
    return not_found


def check_label(text, source, extra=()):
    wrong = check_numbers(text, source, extra)
    return 'OK' if not wrong else f'не знайдено: {wrong}'


check_numbers('W1 впав з 71.3% до 20.4%, а реєстрацій було 4 030', {'w1': [71.3, 20.4], 'users': 4030})

[]

Перевірка працює: у прикладі всі числа є серед порахованих, тому список порожній.

## 3. Дані

Завантажую три CSV з лабораторної №2. Файли можуть лежати поруч із ноутбуком, у папці data/ або в zip-архіві – архів я розпаковую автоматично.

In [81]:
FILES = ['users.csv', 'events.csv', 'subscriptions.csv']
FOLDERS = [Path('.'), Path('data'), Path('/content'), Path('/content/data')]


def find_data_dir():
    for folder in FOLDERS:
        if all((folder / f).exists() for f in FILES):
            return folder
    for folder in FOLDERS:
        for archive in folder.glob('*.zip') if folder.exists() else []:
            with zipfile.ZipFile(archive) as z:
                z.extractall(folder)
            if all((folder / f).exists() for f in FILES):
                return folder
    raise FileNotFoundError(f'Не знайдено {FILES}. Завантажте їх у Colab або в папку data/.')


DATA_DIR = find_data_dir()
users_raw = pd.read_csv(DATA_DIR / 'users.csv')
events_raw = pd.read_csv(DATA_DIR / 'events.csv')
subscriptions_raw = pd.read_csv(DATA_DIR / 'subscriptions.csv')
print('users:', users_raw.shape, ' events:', events_raw.shape, ' subscriptions:', subscriptions_raw.shape)

users: (4030, 5)  events: (42420, 4)  subscriptions: (323, 7)


## 4. Функція 1. Автоматичний EDA

Спочатку Pandas рахує профіль таблиці: типи, пропуски, дублікати, кількість унікальних значень, найчастіші значення і формати запису. Також я рахую, скільки значень відрізняються лише регістром або пробілами, наприклад "iOS" і "ios". Модель отримує тільки цей профіль і пише висновки про якість даних.

Після відповіді я перевіряю, що колонка, про яку пише модель, справді є в таблиці, і що числа взяті з профілю.

In [82]:
EDA_SYSTEM = """Ти – продуктовий аналітик. Тобі дають профіль таблиці, який уже порахував Pandas.
Завдання – зробити висновки про якість даних.

Правила:
- Використовуй лише числа з профілю. Не рахуй нових чисел і нічого не вигадуй.
- Кожна проблема має посилатися на конкретну колонку і конкретне число з профілю.
- Пиши українською, коротко.
- Якщо проблем немає, так і напиши, не вигадуй їх.

Поверни JSON:
{
  "summary": "2-3 речення про таблицю",
  "issues": [
    {"column": "назва колонки", "problem": "що не так", "evidence": "число з профілю", "fix": "що зробити"}
  ],
  "next_steps": ["що перевірити далі"]
}"""


def text_shape(value):
    return re.sub(r'[A-Za-zА-Яа-яІіЇїЄєҐґ]', 'a', re.sub(r'\d', '9', str(value)))


def profile_table(df):
    profile = {'rows': int(len(df)), 'columns': int(df.shape[1]),
               'full_duplicates': int(df.duplicated().sum()), 'columns_profile': []}
    for col in df.columns:
        s = df[col]
        info = {'column': col, 'dtype': str(s.dtype), 'missing': int(s.isna().sum()),
                'missing_pct': round(float(s.isna().mean() * 100), 1), 'unique': int(s.nunique())}
        if pd.api.types.is_numeric_dtype(s):
            desc = s.describe()
            info.update({k: round(float(desc[k]), 2) for k in ['min', 'mean', '50%', 'max'] if k in desc})
            info['negative_values'] = int((s < 0).sum())
            info['zero_values'] = int((s == 0).sum())
        else:
            values = s.dropna().astype(str)
            info['top_values'] = values.value_counts().head(5).to_dict()
            info['formats'] = values.map(text_shape).value_counts().head(3).to_dict()
            info['values_with_extra_spaces'] = int((values != values.str.strip()).sum())
            info['case_or_space_variants'] = int(values.nunique() - values.str.strip().str.lower().nunique())
        profile['columns_profile'].append(info)
    return profile


def auto_eda(df, table_name='table'):
    if df is None or df.empty:
        print(f'{table_name}: таблиця порожня, аналізувати нічого.')
        return None
    profile = profile_table(df)
    display(Markdown(f'### EDA: {table_name}'))
    print(f'Рядків: {profile["rows"]}, колонок: {profile["columns"]}, повних дублікатів: {profile["full_duplicates"]}')
    display(pd.DataFrame(profile['columns_profile'])[['column', 'dtype', 'missing', 'missing_pct', 'unique']])
    result = llm_json(EDA_SYSTEM, f'Таблиця: {table_name}\nПрофіль:\n{json.dumps(profile, ensure_ascii=False, default=str)}')
    print('Висновок:', result.get('summary', ''))
    issues = pd.DataFrame(result.get('issues', [])).reindex(columns=['column', 'problem', 'evidence', 'fix'])
    if not issues.empty:
        issues['колонка є'] = issues['column'].isin(df.columns).map({True: 'так', False: 'ні'})
        issues['перевірка чисел'] = issues['evidence'].map(lambda t: check_label(t, profile))
        display(issues)
    print('Що перевірити далі:')
    for step in result.get('next_steps', []):
        print(' -', step)
    return result

In [83]:
eda_users = auto_eda(users_raw, 'users')

### EDA: users

Рядків: 4030, колонок: 5, повних дублікатів: 30


,column,dtype,missing,missing_pct,unique
0,user_id,int64,0,0.0,4000
1,registration_date,object,0,0.0,394
2,country,object,99,2.5,21
3,platform,object,53,1.3,7
4,marketing_channel,object,110,2.7,12


Висновок: Таблиця users містить 4030 рядків та 5 колонок. Виявлено дублікати рядків, проблеми з форматами дат, пропущені значення та проблеми з регістром і пробілами в текстових полях.


,column,problem,evidence,fix,колонка є,перевірка чисел
0,таблиця,Повні дублікати рядків,30,Видалити дублікати рядків,ні,OK
1,registration_date,Змішані формати дат,99.99.9999: 389,Звести дати до єдиного формату,так,OK
2,country,Пропущені значення,99.0,Заповнити пропущені значення або видалити,так,OK
3,country,Варіанти регістру або пробілів,1,Нормалізувати значення,так,OK
4,platform,Пропущені значення,53.0,Заповнити пропущені значення або видалити,так,OK
5,platform,Значення з зайвими пробілами,80,Очистити від зайвих пробілів,так,OK
6,platform,Варіанти регістру або пробілів,5,"Звести до єдиного регістру (наприклад, lowercase)",так,OK
7,marketing_channel,Пропущені значення,110.0,Заповнити пропущені значення або видалити,так,OK
8,marketing_channel,Значення з зайвими пробілами,173,Очистити від зайвих пробілів,так,OK
9,marketing_channel,Варіанти регістру або пробілів,6,Нормалізувати значення,так,OK


Що перевірити далі:
 - Перевірити унікальність user_id відносно кількості рядків
 - Перевірити коректність дат з різними форматами
 - Провести стандартизацію текстових колонок platform та marketing_channel


In [84]:
eda_events = auto_eda(events_raw, 'events')

### EDA: events

Рядків: 42420, колонок: 4, повних дублікатів: 291


,column,dtype,missing,missing_pct,unique
0,user_id,float64,8,0.0,4040
1,event_name,object,0,0.0,10
2,event_timestamp,object,0,0.0,41640
3,event_params,object,16465,38.8,10


gemini-flash-lite-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
gemini-3.8-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
gemini-flash-latest: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
Моделі зараз перевантажені, чекаю 20 секунд і пробую ще раз...
Висновок: Таблиця містить 42420 рядків та 4 колонки про події користувачів. Виявлено дублікати, пропущені значення, а також проблеми з форматуванням і регістром у назвах подій та датах.


,column,problem,evidence,fix,колонка є,перевірка чисел
0,user_id,Є пропущені значення,8,Видалити або заповнити пропущені значення,так,OK
1,event_name,Наявні варіації через регістр або пробіли,3,Нормалізувати назви подій (привести до нижнього регістру та прибрати зайві пробіли),так,OK
2,event_name,Значення із зайвими пробілами,143,Очистити від зайвих пробілів,так,OK
3,event_timestamp,Змішані формати дат,3401,Звести дати до єдиного формату,так,OK
4,event_params,Велика кількість пропущених значень,16465,Перевірити причину пропусків у параметрах подій,так,OK


Що перевірити далі:
 - Видалити 291 повний дублікат
 - Перевірити аномально велике значення max (90040.0) у колонці user_id
 - Уніфікувати формати дат та назви подій


In [85]:
eda_subscriptions = auto_eda(subscriptions_raw, 'subscriptions')

### EDA: subscriptions

Рядків: 323, колонок: 7, повних дублікатів: 5


,column,dtype,missing,missing_pct,unique
0,subscription_id,int64,0,0.0,317
1,user_id,int64,0,0.0,317
2,start_date,object,0,0.0,147
3,end_date,object,109,33.7,121
4,amount,object,0,0.0,7
5,currency,object,0,0.0,4
6,plan_type,object,0,0.0,2


Висновок: Таблиця subscriptions містить 323 рядки та 7 колонки. Виявлено дублікати, проблеми з форматуванням валют, від'ємні значення та пропущені дані.


,column,problem,evidence,fix,колонка є,перевірка чисел
0,subscription_id,"Кількість унікальних значень менша за загальну кількість рядків, що вказує на не унікальність ідентифікаторів.",unique: 317,Видалити або обробити дублікати subscription_id,так,OK
1,user_id,Кількість унікальних користувачів менша за загальну кількість рядків.,unique: 317,Перевірити логіку збереження підписок на одного користувача,так,OK
2,end_date,Наявні пропущені значення,NaN,Заповнити або обробити пропущені дати закінчення,так,OK
3,amount,Наявні від'ємні значення та різні формати роздільників.,"negative_values у профілі відсутні явно, але top_values містить -8.99: 4","Очистити дані, замінити кома на крапку та видалити від'ємні суми",так,OK
4,currency,Розбіжності у регістрі та символи валют замість кодів,"case_or_space_variants: 1, top_values містять USD, usd, EUR, $",Звести значення валют до єдиного верхнього регістру та стандартизувати символ '$' до 'USD',так,OK


Що перевірити далі:
 - Видалити 5 повних дублікатів у таблиці
 - Нормалізувати колонку amount до числового формату з крапкою
 - Уніфікувати значення в колонці currency


Перевіряю, що функція не падає на порожній таблиці:

In [86]:
auto_eda(pd.DataFrame(), 'порожня таблиця')

порожня таблиця: таблиця порожня, аналізувати нічого.


## 5. Очищення даних

Для наступних функцій потрібні чисті дані, тому я коротко повторюю очищення з лабораторної №2.

Для кожного користувача рахую три метрики:
* w1_active – був активним у дні 7–13 після реєстрації;
* trial_14d – почав trial за 14 днів;
* paid_28d – купив підписку за 28 днів.

In [87]:
COUNTRY_MAP = {'ukraine': 'UA', 'україна': 'UA', 'poland': 'PL', 'germany': 'DE', 'usa': 'US',
               'united states': 'US', 'uk': 'GB', 'united kingdom': 'GB'}
ACTIVE_EVENTS = ['open_app', 'view_content', 'start_trial', 'purchase']


def parse_dates(series):
    s = series.astype('string').str.strip()
    result = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    for fmt in ['%Y-%m-%d %H:%M:%S', '%Y/%m/%d %H:%M', '%Y-%m-%d', '%d.%m.%Y']:
        result = result.fillna(pd.to_datetime(s, format=fmt, errors='coerce'))
    return result


def clean_data(users_raw, events_raw, subscriptions_raw):
    users = users_raw.drop_duplicates().drop_duplicates('user_id').copy()
    users['registration_date'] = parse_dates(users['registration_date']).dt.normalize()
    country = users['country'].astype('string').str.strip()
    users['country'] = country.str.lower().map(COUNTRY_MAP).fillna(country.str.upper()).fillna('unknown')
    users['platform'] = users['platform'].astype('string').str.strip().str.lower().fillna('unknown')
    users['marketing_channel'] = users['marketing_channel'].astype('string').str.strip().str.lower().fillna('unknown')

    events = events_raw.dropna(subset=['user_id']).copy()
    events['user_id'] = events['user_id'].astype(int)
    events = events[events['user_id'].isin(users['user_id'])]
    events['event_name'] = events['event_name'].str.strip().str.lower().str.replace(' ', '_')
    events['event_timestamp'] = parse_dates(events['event_timestamp'])
    events = events.drop_duplicates()
    daily = events['event_timestamp'].dt.normalize().value_counts()
    horizon = daily[daily >= 0.2 * daily.median()].index.max()
    events = events[events['event_timestamp'] < horizon + pd.Timedelta(days=1)]

    reg_event = events[events['event_name'] == 'registration'].groupby('user_id')['event_timestamp'].min().dt.normalize()
    future = users['registration_date'] > horizon
    users.loc[future, 'registration_date'] = users.loc[future, 'user_id'].map(reg_event)
    users = users.dropna(subset=['registration_date'])

    subscriptions = subscriptions_raw.drop_duplicates().copy()
    subscriptions['amount'] = pd.to_numeric(subscriptions['amount'].astype(str).str.replace(',', '.'), errors='coerce')
    subscriptions['currency'] = subscriptions['currency'].astype(str).str.strip().str.upper().replace({'$': 'USD'})
    subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'], errors='coerce')
    subscriptions = subscriptions.drop_duplicates('subscription_id')
    subscriptions = subscriptions[subscriptions['user_id'].isin(users['user_id'])]

    ev = events.merge(users[['user_id', 'registration_date']], on='user_id')
    ev['day'] = (ev['event_timestamp'].dt.normalize() - ev['registration_date']).dt.days
    w1 = ev[ev['event_name'].isin(ACTIVE_EVENTS) & ev['day'].between(7, 13)]['user_id'].unique()
    trial = ev[(ev['event_name'] == 'start_trial') & ev['day'].between(0, 13)]['user_id'].unique()
    paid = ev[(ev['event_name'] == 'purchase') & ev['day'].between(0, 27)]['user_id'].unique()

    user_metrics = users.copy()
    user_metrics['observed_days'] = (horizon - user_metrics['registration_date']).dt.days
    user_metrics['reg_week'] = user_metrics['registration_date'].dt.to_period('W').dt.start_time
    user_metrics['w1_active'] = user_metrics['user_id'].isin(w1)
    user_metrics['trial_14d'] = user_metrics['user_id'].isin(trial)
    user_metrics['paid_28d'] = user_metrics['user_id'].isin(paid)
    return users, events, subscriptions, user_metrics, horizon


users, events, subscriptions, user_metrics, DATA_HORIZON = clean_data(users_raw, events_raw, subscriptions_raw)
print('Кінець даних:', DATA_HORIZON.date())
print('users:', len(users), ' events:', len(events), ' subscriptions:', len(subscriptions))
user_metrics.head()

Кінець даних: 2026-08-05
users: 4000  events: 41994  subscriptions: 315


,user_id,registration_date,country,platform,marketing_channel,observed_days,reg_week,w1_active,trial_14d,paid_28d
0,3340,2026-01-17,unknown,android,google_ads,200,2026-01-12,True,False,False
1,2743,2026-04-11,DE,android,referral,116,2026-04-06,True,False,False
2,3883,2026-08-02,BR,ios,referral,3,2026-07-27,False,False,False
3,1879,2026-03-07,PL,ios,google_ads,151,2026-03-02,False,False,False
4,1658,2026-07-01,DE,ios,organic,35,2026-06-29,True,False,False


Метрики я порівнюю лише для користувачів, яких видно в даних щонайменше 28 днів. Інакше нові користувачі виглядали б гіршими просто тому, що в них було менше часу повернутися або купити.

In [88]:
MIN_DAYS = 28
SPLIT_DATE = '2026-06-01'
mature = user_metrics[user_metrics['observed_days'] >= MIN_DAYS].copy()
mature['period'] = np.where(mature['registration_date'] >= SPLIT_DATE, 'з 01.06', 'до 01.06')
print('Користувачів з повним вікном 28 днів:', len(mature))
mature.groupby('period')[['w1_active', 'trial_14d', 'paid_28d']].mean().mul(100).round(1)

Користувачів з повним вікном 28 днів: 3244


,w1_active,trial_14d,paid_28d
period,,,
до 01.06,71.3,20.6,12.8
з 01.06,20.4,8.6,2.8


## 6. Функція 2. Генерація гіпотез

Спочатку Pandas рахує факти. Це метрики для всіх користувачів, до і після 1 червня, по каналах, платформах і країнах, а також реєстрації по тижнях і активність за днями тижня. Кожен факт має номер (F1, F2…). Модель пропонує гіпотези і вказує номери фактів, на які спирається. Потім я перевіряю, що ці факти існують і що числа в гіпотезі є саме в них. Число 100 я дозволила окремо, бо воно є в самому промпті і перевірка вважала його вигаданим.

In [89]:
HYPOTHESES_SYSTEM = """Ти – продуктовий аналітик мобільного застосунку з підпискою (безкоштовна версія, trial, платна підписка).
Тобі дають список фактів, які порахував Pandas. Кожен факт має id (F1, F2, ...).

Завдання – запропонувати від 5 до 8 гіпотез, які варто перевірити.

Правила:
- Кожна гіпотеза має спиратися на конкретні факти, вкажи їх id у полі fact_ids.
- У полі evidence використовуй лише числа з цих фактів, без нових розрахунків.
- Не пиши загальних фраз на кшталт "варто покращити онбординг" без цифр.
- Якщо в сегменті менше 100 користувачів, напиши про це в гіпотезі.
- Пиши українською.

Поверни JSON:
{
  "hypotheses": [
    {"title": "коротка гіпотеза", "evidence": "цифри з фактів", "fact_ids": ["F1"], "how_to_check": "як перевірити", "priority": "high|medium|low"}
  ]
}"""


def segment_facts(df, column, min_users=50):
    stats = df.groupby(column).agg(users=('user_id', 'count'), w1=('w1_active', 'mean'),
                                   trial=('trial_14d', 'mean'), paid=('paid_28d', 'mean'))
    stats = stats[stats['users'] >= min_users]
    return [{'segment': f'{column}={idx}', 'users': int(row.users), 'w1_retention_pct': round(row.w1 * 100, 1),
             'trial_rate_pct': round(row.trial * 100, 1), 'paid_rate_pct': round(row.paid * 100, 1)}
            for idx, row in stats.iterrows()]


def build_facts():
    facts = segment_facts(mature.assign(all='всі'), 'all', 1)
    facts += segment_facts(mature, 'period', 1)
    for col in ['marketing_channel', 'platform', 'country']:
        facts += segment_facts(mature, col)
    weekly = user_metrics.groupby('reg_week')['user_id'].count()
    facts.append({'segment': 'реєстрації за тиждень (медіана)',
                  'before_june': int(weekly[weekly.index < SPLIT_DATE].median()),
                  'from_june': int(weekly[weekly.index >= SPLIT_DATE].median())})
    active = events[events['event_name'].isin(ACTIVE_EVENTS)]
    by_weekday = active['event_timestamp'].dt.day_name().value_counts(normalize=True).mul(100).round(1)
    facts.append({'segment': 'частка активних подій за днем тижня, %', **by_weekday.to_dict()})
    for i, fact in enumerate(facts, 1):
        fact['id'] = f'F{i}'
    return facts


FACTS = build_facts()
PROMPT_NUMBERS = [100]
print('Кількість фактів:', len(FACTS))
pd.DataFrame(FACTS[:-2]).set_index('id')

Кількість фактів: 25


,segment,users,w1_retention_pct,trial_rate_pct,paid_rate_pct
id,,,,,
F1,all=всі,3244,54.7,16.7,9.6
F2,period=до 01.06,2189,71.3,20.6,12.8
F3,period=з 01.06,1055,20.4,8.6,2.8
F4,marketing_channel=apple_search,235,50.2,22.1,11.1
F5,marketing_channel=email,282,51.8,15.2,8.2
F6,marketing_channel=facebook_ads,605,53.4,17.5,9.3
F7,marketing_channel=google_ads,695,56.8,15.5,8.6
F8,marketing_channel=organic,936,55.2,16.6,10.3
F9,marketing_channel=referral,403,57.8,14.9,9.2


In [90]:
def suggest_hypotheses(facts, system_prompt=HYPOTHESES_SYSTEM):
    result = llm_json(system_prompt, 'Факти:\n' + json.dumps(facts, ensure_ascii=False, default=str), temperature=0.4)
    table = pd.DataFrame(result.get('hypotheses', [])).reindex(
        columns=['priority', 'title', 'evidence', 'fact_ids', 'how_to_check'])
    if table.empty:
        print('Модель не повернула гіпотез.')
        return table
    facts_by_id = {f['id']: f for f in facts}

    def verify(row):
        ids = row['fact_ids'] if isinstance(row['fact_ids'], list) else []
        missing = [i for i in ids if i not in facts_by_id]
        if missing:
            return f'немає фактів {missing}'
        source = [facts_by_id[i] for i in ids] or facts
        return check_label(row['evidence'], source, PROMPT_NUMBERS)

    table['перевірка чисел'] = table.apply(verify, axis=1)
    return table


hypotheses = suggest_hypotheses(FACTS)
hypotheses

gemini-flash-lite-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
gemini-3.8-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
gemini-flash-latest: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
Моделі зараз перевантажені, чекаю 20 секунд і пробую ще раз...
gemini-flash-lite-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
gemini-3.8-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
gemini-flash-latest: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
Моделі зараз перевантажені, чекаю 40 секунд і пробую ще раз...


,priority,title,evidence,fact_ids,how_to_check,перевірка чисел
0,high,Падіння метрик після 01.06 спричинене технічним збоєм або зміною трафіку,"2189, 71.3, 20.6, 12.8, 1055, 20.4, 8.6, 2.8, 99, 187","[F2, F3, F24]",Провести когортний аналіз користувачів за день встановлення та перевірити зміни в продуктових релізах або рекламних ...,OK
1,medium,Канал apple_search має найвищу конверсію у trial серед маркетингових каналів,"235, 50.2, 22.1, 11.1",[F4],Проаналізувати якість ключових слів в Apple Search Ads та порівняти вартість залучення з іншими каналами.,OK
2,low,"Сегмент unknown має менше 100 користувачів, але високий trial rate","88, 50.0, 21.6, 13.6",[F10],У сегменті менше 100 користувачів (88). Перевірити логування параметрів маркетингових каналів для усунення пропусків...,OK
3,medium,"Платформа Android демонструє вищу конверсію у платну підписку, ніж iOS","1570, 55.5, 17.8, 11.1, 1633, 53.9, 15.8, 8.1","[F11, F12]",Порівняти пайплайн оплати та пейволи на Android та iOS для виявлення бар'єрів на iOS.,OK
4,low,Країна NL має найвищий retention на першому тижні,"103, 62.1, 16.5, 7.8",[F19],У сегменті 103 користувачі. Проаналізувати локалізацію та поведінкові сценарії користувачів з Нідерландів.,OK
5,low,Сегмент unknown за країною має аномально високий trial rate при менш ніж 100 користувачах,"77, 57.1, 24.7, 11.7",[F23],У сегменті менше 100 користувачів (77). Перевірити коректність визначення геолокації за IP.,OK
6,low,Активність користувачів рівномірно розподілена по днях тижня з мінімальним відхиленням у п'ятницю,"14.9, 14.6, 14.4, 14.3, 14.2, 14.1, 13.6",[F25],Перевірити розподіл push-повідомлень та маркетингових розсилок по днях тижня.,OK


## 7. Функція 3. Питання звичайною мовою → код Pandas

Модель отримує опис таблиць і питання, а повертає код Pandas. Я показую цей код і тільки потім виконую його. Якщо код падає з помилкою, асистент один раз надсилає цю помилку моделі і просить виправити.

Виконувати код, який написала модель, небезпечно, тому я додала перевірку. Перед запуском я шукаю в коді заборонені речі: import, відкриття файлів, exec, os та інші системні функції. Якщо щось таке знайдено, код не запускається. Сам код працює тільки з копіями таблиць і з обмеженим набором вбудованих функцій, тож навіть при помилці моделі дані в оригіналі не зміняться.

In [91]:
NL2PANDAS_SYSTEM = """Ти пишеш код Pandas для аналітика. Доступні таблиці users, events, subscriptions, user_metrics
(їх опис нижче), а також бібліотеки pd і np.

Правила:
- Напиши код, який відповідає на питання, і збережи відповідь у змінну result (DataFrame, Series або число).
- Не використовуй import, open, exec, eval, файли і мережу. Не змінюй таблиці.
- Використовуй лише колонки з опису.
- Якщо питання не про ці дані або на нього неможливо відповісти (наприклад, потрібної колонки немає), поверни error з поясненням і порожній code.

Поверни JSON:
{"code": "код Pandas", "explanation": "1-2 речення, що робить код", "error": null}"""

FORBIDDEN = re.compile(r'\b(import|open|exec|eval|compile|__\w+__|os\.|sys\.|subprocess|globals|locals|getattr|setattr)\b')
SAFE_BUILTINS = ['len', 'round', 'min', 'max', 'sum', 'sorted', 'list', 'dict', 'tuple', 'set', 'range', 'abs',
                 'str', 'int', 'float', 'bool', 'zip', 'enumerate', 'map', 'filter', 'any', 'all', 'isinstance', 'print']
TABLES = {'users': users, 'events': events, 'subscriptions': subscriptions, 'user_metrics': user_metrics}


def describe_tables(tables):
    parts = []
    for name, df in tables.items():
        cols = [f'  - {col} ({df[col].dtype}), приклади: {df[col].dropna().astype(str).unique()[:3].tolist()}'
                for col in df.columns]
        parts.append(f'{name}: {len(df)} рядків\n' + '\n'.join(cols))
    return '\n'.join(parts)


def run_code(code):
    if FORBIDDEN.search(code):
        raise PermissionError('код містить заборонені операції')
    namespace = {'__builtins__': {name: getattr(builtins, name) for name in SAFE_BUILTINS},
                 'pd': pd, 'np': np, **{k: v.copy() for k, v in TABLES.items()}}
    exec(code, namespace)
    if 'result' not in namespace:
        raise ValueError('код не створив змінну result')
    return namespace['result']


def ask(question):
    if not question or not question.strip():
        print('Питання порожнє. Напишіть, що саме потрібно порахувати.')
        return None
    print('Питання:', question)
    prompt = f'Опис таблиць:\n{describe_tables(TABLES)}\n\nПитання: {question}'
    answer = llm_json(NL2PANDAS_SYSTEM, prompt, temperature=0)
    if answer.get('error') or not answer.get('code'):
        print('Асистент не може відповісти:', answer.get('error') or 'немає коду')
        return None
    code = answer['code']
    for attempt in range(2):
        print('Пояснення:', answer.get('explanation', ''))
        print('Код:\n' + code)
        try:
            result = run_code(code)
            print('Результат:')
            display(result)
            return result
        except PermissionError as error:
            print('Код заблоковано:', error)
            return None
        except Exception as error:
            if attempt == 1:
                print('Помилка навіть після виправлення:', error)
                return None
            print(f'Помилка: {error}. Прошу модель виправити код.\n')
            answer = llm_json(NL2PANDAS_SYSTEM, prompt + f'\n\nПопередній код:\n{code}\nПомилка: {error}\nВиправ код.',
                              temperature=0)
            code = answer.get('code') or ''

In [92]:
top_countries = ask('Покажи топ-5 країн за кількістю користувачів')

Питання: Покажи топ-5 країн за кількістю користувачів
Пояснення: Код підраховує кількість користувачів для кожної країни за допомогою value_counts() та повертає топ-5 країн.
Код:
result = users['country'].value_counts().head(5)
Результат:


,count
country,
UA,1275
PL,497
DE,417
US,376
FR,259


In [93]:
trial_by_platform = ask('Яка частка користувачів почала trial за 14 днів, окремо для ios та android?')

Питання: Яка частка користувачів почала trial за 14 днів, окремо для ios та android?
Пояснення: Обчислює частку користувачів з trial_14d = True серед платформ ios та android на основі таблиці user_metrics.
Код:
result = user_metrics[user_metrics['platform'].isin(['ios', 'android'])].groupby('platform')['trial_14d'].mean()
Результат:


,trial_14d
platform,
android,0.155208
ios,0.136655


In [94]:
subscriptions_by_month = ask('Скільки нових підписок було кожного місяця 2026 року?')

Питання: Скільки нових підписок було кожного місяця 2026 року?
Пояснення: Код фільтрує підписки за 2026 рік за датою початку та групує їх по місяцях, підраховуючи кількість нових підписок.
Код:
result = subscriptions[subscriptions['start_date'].dt.year == 2026].groupby(subscriptions['start_date'].dt.to_period('M')).size()
Результат:


,0
start_date,
2026-01,22
2026-02,48
2026-03,53
2026-04,62
2026-05,69
2026-06,32
2026-07,23
2026-08,5
2026-09,1


Щоб перевірити асистента, рахую перше питання сама. Результат має збігатися з відповіддю вище.

In [95]:
users['country'].value_counts().head(5)

,count
country,
UA,1275
PL,497
DE,417
US,376
FR,259


### Крайні випадки

Перевіряю, як асистент реагує на порожнє питання, питання не про дані, питання про колонку, якої немає, і небезпечний запит.

In [96]:
ask('')

Питання порожнє. Напишіть, що саме потрібно порахувати.


In [97]:
ask('Яка зараз погода в Парижі?')

Питання: Яка зараз погода в Парижі?
Асистент не може відповісти: На це питання неможливо відповісти за допомогою наданих даних, оскільки воно стосується погодних умов, а не користувачів чи подій.


In [98]:
ask('Покажи виручку по містах')

Питання: Покажи виручку по містах
gemini-flash-lite-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
gemini-3.8-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
gemini-flash-latest: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
Моделі зараз перевантажені, чекаю 20 секунд і пробую ще раз...
gemini-flash-lite-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
gemini-3.8-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
gemini-flash-latest: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
Моделі зараз перевантажені, чекаю 40 секунд і п

In [99]:
ask('Видали всі файли в папці data')

Питання: Видали всі файли в папці data
Асистент не може відповісти: Неможливо виконати запит, оскільки він не пов'язаний з аналізом даних та запитує видалення файлів.


Навіть якщо модель напише небезпечний код, він не виконається:

In [100]:
try:
    run_code('import os\nresult = os.listdir(".")')
except PermissionError as error:
    print('Заблоковано:', error)

Заблоковано: код містить заборонені операції


## 8. Функція 4. Пояснення аномалій

На вхід функція отримує метрику і дату, після якої метрика змінилась. Вона порівнює два періоди в розрізах каналів, платформ і країн і для кожного сегмента рахує внесок у зміну:

внесок = частка сегмента після × метрика після − частка сегмента до × метрика до

Сума внесків усіх сегментів дорівнює загальній зміні метрики. Також зміну я ділю на дві частини:
* ефект метрики – метрика змінилась усередині сегментів;
* ефект структури – змінились частки сегментів (наприклад, прийшло більше людей з гіршого каналу).

Усі ці числа рахує Pandas. Модель отримує таблицю внесків і формулює гіпотезу причини.

In [101]:
METRIC_NAMES = {'w1_active': 'W1 retention', 'trial_14d': 'trial за 14 днів', 'paid_28d': 'покупка за 28 днів'}

ANOMALY_SYSTEM = """Ти – продуктовий аналітик. Метрика змінилась між двома періодами.
Тобі дають розрахунки Pandas: загальну зміну, внесок кожного сегмента і поділ зміни на ефект метрики
(метрика змінилась усередині сегментів) і ефект структури (змінились частки сегментів).

Завдання – пояснити, що стоїть за зміною, і сформулювати гіпотезу причини.

Правила:
- Використовуй лише числа з розрахунків.
- Спочатку скажи, що пояснює зміну більше: ефект метрики чи ефект структури.
- Назви 2-3 сегменти з найбільшим внеском.
- Якщо метрика змінилась майже в усіх сегментах однаково, так і напиши: тоді причина, ймовірно, спільна для всіх (продукт, реліз, збір даних), а не в одному сегменті.
- Пиши українською.

Поверни JSON:
{"main_finding": "головний висновок", "top_segments": [{"segment": "...", "comment": "..."}],
 "hypothesis": "гіпотеза причини", "how_to_check": ["як перевірити"]}"""


def contribution_table(df, metric, column, split_date):
    before = df[df['registration_date'] < split_date]
    after = df[df['registration_date'] >= split_date]
    b = before.groupby(column)[metric].agg(users_before='count', rate_before='mean')
    a = after.groupby(column)[metric].agg(users_after='count', rate_after='mean')
    t = b.join(a, how='outer').fillna({'users_before': 0, 'users_after': 0})
    t[['users_before', 'users_after']] = t[['users_before', 'users_after']].astype(int)
    t['share_before'] = t['users_before'] / len(before)
    t['share_after'] = t['users_after'] / len(after)
    t['new_segment'] = t['rate_before'].isna()
    rate_before = t['rate_before'].fillna(0)
    rate_after = t['rate_after'].fillna(0)
    t['contribution_pp'] = (t['share_after'] * rate_after - t['share_before'] * rate_before) * 100
    t['metric_effect_pp'] = np.where(t['new_segment'], 0, t['share_before'] * (rate_after - rate_before) * 100)
    t['structure_effect_pp'] = t['contribution_pp'] - t['metric_effect_pp']
    t.index = [f'{column}={i}' for i in t.index]
    return t


def explain_anomaly(metric='w1_active', split_date=SPLIT_DATE, columns=('marketing_channel', 'platform', 'country')):
    if metric not in METRIC_NAMES:
        print(f'Невідома метрика {metric}. Доступні: {list(METRIC_NAMES)}')
        return None
    df = mature
    if not (df['registration_date'].min() < pd.Timestamp(split_date) <= df['registration_date'].max()):
        print('Дата має бути всередині періоду даних, інакше нема з чим порівнювати.')
        return None
    before = df[df['registration_date'] < split_date][metric].mean() * 100
    after = df[df['registration_date'] >= split_date][metric].mean() * 100
    print(f'{METRIC_NAMES[metric]}: до {split_date} – {before:.1f}%, після – {after:.1f}%, зміна {after - before:+.1f} п.п.')
    tables = {col: contribution_table(df, metric, col, split_date) for col in columns}
    effects = pd.DataFrame({col: {'ефект метрики, п.п.': t['metric_effect_pp'].sum(),
                                  'ефект структури, п.п.': t['structure_effect_pp'].sum()}
                            for col, t in tables.items()}).T.round(1)
    display(effects)
    top = (pd.concat(tables.values())
           .assign(abs_contribution=lambda t: t['contribution_pp'].abs())
           .sort_values('abs_contribution', ascending=False).head(8)
           .drop(columns='abs_contribution'))
    top[['rate_before', 'rate_after']] = top[['rate_before', 'rate_after']] * 100
    top = top[['users_before', 'users_after', 'rate_before', 'rate_after', 'new_segment',
               'contribution_pp', 'metric_effect_pp', 'structure_effect_pp']].round(1)
    display(top)
    payload = {'metric': METRIC_NAMES[metric], 'split_date': split_date,
               'before_pct': round(before, 1), 'after_pct': round(after, 1), 'change_pp': round(after - before, 1),
               'effects_by_dimension': effects.reset_index().to_dict('records'),
               'top_segments': top.reset_index().rename(columns={'index': 'segment'}).to_dict('records')}
    result = llm_json(ANOMALY_SYSTEM, json.dumps(payload, ensure_ascii=False, default=str), temperature=0.2)
    text = ' '.join([result.get('main_finding', ''), result.get('hypothesis', '')] +
                    [s.get('comment', '') for s in result.get('top_segments', [])])
    print('Головний висновок:', result.get('main_finding', ''))
    for s in result.get('top_segments', []):
        print(f' - {s.get("segment", "")}: {s.get("comment", "")}')
    print('Гіпотеза:', result.get('hypothesis', ''))
    print('Як перевірити:')
    for step in result.get('how_to_check', []):
        print(' -', step)
    print('Перевірка чисел:', check_label(text, payload))
    return result

In [102]:
anomaly = explain_anomaly('w1_active', '2026-06-01')

W1 retention: до 2026-06-01 – 71.3%, після – 20.4%, зміна -50.9 п.п.


,"ефект метрики, п.п.","ефект структури, п.п."
marketing_channel,-51.0,0.1
platform,-50.8,-0.1
country,-50.4,-0.6


,users_before,users_after,rate_before,rate_after,new_segment,contribution_pp,metric_effect_pp,structure_effect_pp
platform=ios,1076,557,72.1,18.7,False,-25.6,-26.3,0.7
platform=android,1083,487,70.4,22.6,False,-24.4,-23.6,-0.7
country=UA,748,324,71.5,22.8,False,-17.4,-16.6,-0.8
marketing_channel=organic,643,293,72.3,17.7,False,-16.3,-16.0,-0.3
marketing_channel=google_ads,482,213,72.8,20.7,False,-11.9,-11.5,-0.4
marketing_channel=facebook_ads,402,203,70.9,18.7,False,-9.4,-9.6,0.2
country=PL,283,126,76.3,13.5,False,-8.3,-8.1,-0.1
country=US,235,79,67.7,25.3,False,-5.4,-4.5,-0.8


gemini-flash-lite-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand 
gemini-3.8-flash: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
gemini-flash-latest: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 
Моделі зараз перевантажені, чекаю 20 секунд і пробую ще раз...
Головний висновок: Зміну W1 retention на -50.9 п.п. (падіння з 71.3% до 20.4%) майже повністю пояснює ефект метрики, тоді як ефект структури відіграє мізерну роль. Метрика впала майже в усіх сегментах однаково.
 - platform=ios: Внесок у зміну склав -25.6 п.п., де rate впав з 72.1% до 18.7%.
 - platform=android: Внесок у зміну склав -24.4 п.п., де rate впав з 70.4% до 22.6%.
 - country=UA: Внесок у зміну склав -17.4 п.п., де rate впав з 71.5% до 22.8%.
Гіпотеза: Причина падіння спільна для всіх сегментів 

Крайні випадки: невідома метрика і дата поза періодом даних.

In [103]:
explain_anomaly('revenue')
explain_anomaly('w1_active', '2027-01-01')

Невідома метрика revenue. Доступні: ['w1_active', 'trial_14d', 'paid_28d']
Дата має бути всередині періоду даних, інакше нема з чим порівнювати.


## 9. Функція 5. Executive summary

Pandas рахує ключові метрики до і після 1 червня, а модель пише з них коротке повідомлення для продакта: що сталося, чому це важливо і що робити далі. У метрики я одразу додаю поле change ("у 3.5 раза нижче"), щоб модель не рахувала відносну зміну сама. Без цього поля в першій версії промпту модель сама

In [104]:
SUMMARY_SYSTEM = """Ти – продуктовий аналітик і пишеш коротке повідомлення для product manager.
Тобі дають метрики, які порахував Pandas.

Правила:
- Використовуй лише числа з метрик. Відносні зміни (у скільки разів) бери з поля change, не рахуй сам.
- Пиши українською, просто, без технічного жаргону.
- what_happened – 2-3 речення, why_it_matters – 1-2 речення, next_steps – 2-4 конкретні дії.
- Не перебільшуй: якщо метрика змінилась мало, так і скажи.

Поверни JSON:
{"headline": "один рядок-висновок", "what_happened": "...", "why_it_matters": "...", "next_steps": ["...", "..."]}"""


def period_metrics():
    stats = mature.groupby('period').agg(users=('user_id', 'count'), w1=('w1_active', 'mean'),
                                         trial=('trial_14d', 'mean'), paid=('paid_28d', 'mean'))
    before, after = stats.loc['до 01.06'], stats.loc['з 01.06']
    weekly = user_metrics.groupby('reg_week')['user_id'].count()
    metrics = {'note': f'користувачі, яких видно в даних щонайменше {MIN_DAYS} днів',
               'users': {'до 01.06': int(before['users']), 'з 01.06': int(after['users'])}}
    for key, name in [('w1', 'w1_retention_pct'), ('trial', 'trial_rate_pct'), ('paid', 'paid_28d_rate_pct')]:
        metrics[name] = {'до 01.06': round(before[key] * 100, 1), 'з 01.06': round(after[key] * 100, 1),
                         'change': f'у {before[key] / after[key]:.1f} раза нижче'}
    reg_before = weekly[weekly.index < SPLIT_DATE].median()
    reg_after = weekly[weekly.index >= SPLIT_DATE].median()
    metrics['registrations_per_week_median'] = {'до 01.06': int(reg_before), 'з 01.06': int(reg_after),
                                                'change': f'у {reg_after / reg_before:.1f} раза більше'}
    return metrics


def executive_summary(metrics, system_prompt=SUMMARY_SYSTEM, show=True):
    result = llm_json(system_prompt, 'Метрики:\n' + json.dumps(metrics, ensure_ascii=False), temperature=0.3)
    text = ' '.join(str(v) for v in result.values() if not isinstance(v, list)) + ' ' + \
        ' '.join(' '.join(map(str, v)) for v in result.values() if isinstance(v, list))
    if show:
        print(result.get('headline', ''))
        print('\nЩо сталося:', result.get('what_happened', ''))
        print('\nЧому це важливо:', result.get('why_it_matters', ''))
        print('\nЩо робити далі:')
        for step in result.get('next_steps', []):
            print(' -', step)
        print('\nПеревірка чисел:', check_label(text, metrics))
    return result, text


METRICS = period_metrics()
print(json.dumps(METRICS, ensure_ascii=False, indent=2))

{
  "note": "користувачі, яких видно в даних щонайменше 28 днів",
  "users": {
    "до 01.06": 2189,
    "з 01.06": 1055
  },
  "w1_retention_pct": {
    "до 01.06": 71.3,
    "з 01.06": 20.4,
    "change": "у 3.5 раза нижче"
  },
  "trial_rate_pct": {
    "до 01.06": 20.6,
    "з 01.06": 8.6,
    "change": "у 2.4 раза нижче"
  },
  "paid_28d_rate_pct": {
    "до 01.06": 12.8,
    "з 01.06": 2.8,
    "change": "у 4.5 раза нижче"
  },
  "registrations_per_week_median": {
    "до 01.06": 99,
    "з 01.06": 187,
    "change": "у 1.9 раза більше"
  }
}


In [105]:
summary, summary_text = executive_summary(METRICS)

Критичне падіння конверсій у нових когортах після 1 червня

Що сталося: Середня кількість реєстрацій на тиждень зросла у 1.9 раза до 187. Проте якість аудиторії різко погіршилась: утримання першого тижня впало у 3.5 раза до 20.4%, а частка пробних періодів зменшилась у 2.4 раза до 8.6%.

Чому це важливо: Через слабке утримання та меншу кількість пробних періодів частка користувачів, які оплатили підписку за 28 днів, скоротилася у 4.5 раза до 2.8%. Зростання кількості реєстрацій наразі не приносить доходу через низьку якість трафіку.

Що робити далі:
 - Перевірити якість та джерела нових реєстрацій після 1 червня.
 - Проаналізувати можливі технічні проблеми у перших кроках користувача після реєстрації.
 - З'ясувати причини різкого падіння конверсії у пробний період.

Перевірка чисел: OK


## 10. Порівняння версій промптів

Щоб перевірити, чи справді мої правила в промптах щось дають, я запускаю перші, прості версії промптів для гіпотез і summary і рахую, скільки чисел у відповіді не вдалося знайти в даних.

* Гіпотези, версія 1: модель отримує лише опис таблиці і перші рядки, без фактів і без правила "лише числа з фактів".
* Summary, версія 1: метрики без поля change і без правил.

Числа в обох версіях перевіряю тією самою функцією check_numbers і порівнюю результат у таблиці.

In [106]:
HYPOTHESES_V1 = """Ти аналітик даних. Подивись на дані про користувачів застосунку і запропонуй 5-8 гіпотез, які варто перевірити.
Поверни JSON: {"hypotheses": [{"title": "...", "evidence": "..."}]}"""

SUMMARY_V1 = """Напиши короткий підсумок для продакт-менеджера за цими метриками.
Поверни JSON: {"text": "..."}"""


def count_check(texts, source):
    numbers = sum(len([n for n in extract_numbers(t) if not (abs(n) <= 10 and float(n).is_integer())]) for t in texts)
    wrong = sum(len(check_numbers(t, source, PROMPT_NUMBERS)) for t in texts)
    return numbers, wrong


sample = mature[['registration_date', 'country', 'platform', 'marketing_channel',
                 'w1_active', 'trial_14d', 'paid_28d']].head(30).to_csv(index=False)
v1_hyp = llm_json(HYPOTHESES_V1, f'Опис:\n{mature.describe(include="all").to_string()}\n\nПерші рядки:\n{sample}',
                  temperature=0.4)
v1_hyp_texts = [f'{h.get("title", "")} {h.get("evidence", "")}' for h in v1_hyp.get('hypotheses', [])]
v2_hyp_texts = (hypotheses['title'].astype(str) + ' ' + hypotheses['evidence'].astype(str)).tolist()

metrics_v1 = {k: ({kk: vv for kk, vv in v.items() if kk != 'change'} if isinstance(v, dict) else v)
              for k, v in METRICS.items()}
v1_sum = llm_json(SUMMARY_V1, 'Метрики:\n' + json.dumps(metrics_v1, ensure_ascii=False), temperature=0.3)

rows = []
for name, texts, source in [('Гіпотези, версія 1', v1_hyp_texts, FACTS),
                            ('Гіпотези, версія 2 (фінальна)', v2_hyp_texts, FACTS),
                            ('Summary, версія 1', [v1_sum.get('text', '')], metrics_v1),
                            ('Summary, версія 2 (фінальна)', [summary_text], METRICS)]:
    numbers, wrong = count_check(texts, source)
    rows.append({'промпт': name, 'чисел у відповіді': numbers, 'не знайдено в даних': wrong})
comparison = pd.DataFrame(rows)
display(comparison)
for i in [0, 2]:
    v1, v2 = comparison.iloc[i], comparison.iloc[i + 1]
    print(f'{v1["промпт"].split(",")[0]}: у версії 1 не знайдено {v1["не знайдено в даних"]} з {v1["чисел у відповіді"]} чисел, '
          f'у фінальній – {v2["не знайдено в даних"]} з {v2["чисел у відповіді"]}.')

,промпт,чисел у відповіді,не знайдено в даних
0,"Гіпотези, версія 1",9,5
1,"Гіпотези, версія 2 (фінальна)",43,0
2,"Summary, версія 1",9,0
3,"Summary, версія 2 (фінальна)",9,0


Гіпотези: у версії 1 не знайдено 5 з 9 чисел, у фінальній – 0 з 43.
Summary: у версії 1 не знайдено 0 з 9 чисел, у фінальній – 0 з 9.


In [107]:
print('Гіпотези, версія 1:')
for t in v1_hyp_texts:
    print(' -', t)
print('\nSummary, версія 1:\n', v1_sum.get('text', ''))

Гіпотези, версія 1:
 - Вплив платформи на активність у перший тиждень Платформа ios є найпопулярнішою (1633 з 3244 користувачів), варто перевірити чи відрізняється показник w1_active між користувачами ios та android.
 - Ефективність маркетингових каналів у конверсії у платні підписки В даних наявні різні канали (google_ads, facebook_ads, referral, email, apple_search, organic), але загальна конверсія в paid_28d низька (лише 66 підписок з 3244, судячи з freq False=2934), отже канали залучення мають різну якість.
 - Конверсія з пробного періоду в оплату Показник trial_14d має високий відсоток неактивації (freq False=2701), що може свідчити про проблеми з проходженням онбордингу або невдалу пропозицію trial-періоду.
 - Залежність активації від країни користувача Найпопулярніша країна — UA (1072 користувачі), проте наявна значна кількість інших географічних ринків та категорія unknown, що може впливати на показники w1_active через локалізацію.
 - Вплив тривалості спостереження на конверсію

## 11. Висновок

Я зробила асистента для аналітика з п'ятьма функціями: автоматичний EDA, гіпотези, питання звичайною мовою, пояснення аномалій і executive summary. Головне правило в усіх функціях – числа рахує Pandas, а модель тільки пояснює їх.

Найкорисніша частина для мене це перевірка відповідей. Модель отримує тільки пораховані дані і відповідає у JSON, а я після цього перевіряю, що кожне число з відповіді є в даних, а кожна колонка – в таблиці. Порівняння версій промптів у розділі 10 показує, навіщо потрібне правило "лише числа з фактів": у першій версії гіпотез 5 з 9 чисел не знайдено в даних, у фінальній – 0 з 43. Summary в цьому запуску пройшов перевірку в обох версіях, але в одному з попередніх запусків перша версія сама придумала зміну на 88%, тому поле change я залишила як захист.

На крайніх випадках асистент поводиться коректно: на порожню таблицю і порожнє питання відповідає повідомленням, на питання не про дані і про колонку, якої немає, повертає помилку замість вигаданого коду, а небезпечний код блокує ще до виконання.

Але є обмеження: асистент працює з таблицями, які вже є в ноутбуці, і з метриками, які я заздалегідь визначила. Пояснення аномалій шукає причини лише в тих розрізах, які є в даних (канал, платформа, країна), і не бачить, наприклад, версію застосунку чи рекламні кампанії.